In [58]:
import cv2
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt 
import matplotlib.image as mpimg
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow import keras
from keras import Sequential
from keras.layers import *
from tensorflow.keras.losses import BinaryCrossentropy
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.optimizers import Adam , Adamax
from tensorflow.keras.applications import *
from tensorflow.keras.callbacks import EarlyStopping
import warnings 

# warnings.filterwarnings("ignore")

In [59]:
# x = os.listdir('C:/Users/Mark/Desktop/PadChest/0')

# for f in x:
#     print(f)

In [60]:
# Data_Dir
# C:/Users/Mark/Desktop/PadChest
# C:/Users/markf/Desktop/PadChest

# data_dir = 'C:/Users/Mark/Desktop/PadChest/0'  
# csv_file = 'C:/Users/Mark/Desktop/PadChest/padchest_labels.csv'

data_dir = 'C:/Users/markf/Desktop/PadChest/0'
csv_file = 'C:/Users/markf/Desktop/PadChest/padchest_labels.csv'

IMAGE_SIZE = (256,256)
BATCH_SIZE = 32
EPOCHS = 10

# checks if image file exists
def file_exists(image_id):
    image_path = os.path.join(data_dir, image_id)
    return os.path.exists(image_path)

# Safely parse the 'Labels' column and handle errors
def parse_labels(label_entry):
    if isinstance(label_entry, str):  # Ensure the entry is a string
        try:
            return eval(label_entry)  # Evaluate the string
        except Exception as e:
            print(f"Error evaluating label: {label_entry}. Error: {e}")
            return []  # Return an empty list for problematic entries
    return []  # Return an empty list if the entry is not a string

# metadata csv file
df = pd.read_csv(csv_file)

# filter out rows where the image file doesn't exist
df['File_Exists'] = df['ImageID'].apply(file_exists)
df = df[df['File_Exists']].drop(columns=['File_Exists'])

# Apply the safe parsing function
df['Parsed_Labels'] = df['Labels'].apply(parse_labels)

# Get all unique labels from the parsed labels
all_classes = set(label for label_list in df['Parsed_Labels'] for label in label_list)

# Map classes to indices
class_to_index = {class_name: idx for idx, class_name in enumerate(all_classes)}

# One-hot encode the parsed labels
def encode_labels(label_list):
    binary_vector = [0] * len(class_to_index)
    for label in label_list:
        if label in class_to_index:
            binary_vector[class_to_index[label]] = 1
    return binary_vector

# Apply encoding to the Parsed_Labels column
df['Encoded_Labels'] = df['Parsed_Labels'].apply(encode_labels)


# split the data into training, validation, and testing sets
train_df, test_df = train_test_split(df, test_size=0.2, random_state=999)
train_df, val_df = train_test_split(train_df, test_size=0.1, random_state=999)

In [74]:
# Helper function to preprocess images
def preprocess_image(path, label):
    try:
        # Read with OpenCV instead of TensorFlow
        image = tf.numpy_function(
            lambda x: cv2.imread(x.decode('utf-8'), [path], tf.uint8))
        image = tf.image.resize(image, IMAGE_SIZE)
        image = image / 255.0
    except Exception as e:
        print(f"Error processing {path}: {e}")
        image = tf.zeros((*IMAGE_SIZE, 3))
    
    label = tf.cast(label, tf.float32)
    return image, label

# Helper function to create datasets
def create_dataset(dataframe):
    # Map image paths to full paths
    dataframe['Full_Image_Path'] = dataframe['ImageID'].apply(lambda x: os.path.normpath(os.path.join(data_dir, x)))
    # Create a TensorFlow dataset
    paths = dataframe['Full_Image_Path'].values
    labels = list(dataframe['Encoded_Labels'].values)
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    dataset = dataset.map(preprocess_image).batch(BATCH_SIZE)
    return dataset

In [75]:
# Create datasets
train_ds = create_dataset(train_df)
val_ds = create_dataset(val_df)
test_ds = create_dataset(test_df)

Error processing Tensor("args_0:0", shape=(), dtype=string): Missing required argument: 'Tout'
  If using tf.numpy_function as a decorator, set `Tout`
  **by name** above the function:
  `@tf.numpy_function(Tout=tout)`
Error processing Tensor("args_0:0", shape=(), dtype=string): Missing required argument: 'Tout'
  If using tf.numpy_function as a decorator, set `Tout`
  **by name** above the function:
  `@tf.numpy_function(Tout=tout)`
Error processing Tensor("args_0:0", shape=(), dtype=string): Missing required argument: 'Tout'
  If using tf.numpy_function as a decorator, set `Tout`
  **by name** above the function:
  `@tf.numpy_function(Tout=tout)`


In [ ]:
# train_df['Full_Image_Path']

502     C:\Users\markf\Desktop\PadChest\0\820904313880...
4       C:\Users\markf\Desktop\PadChest\0\113855343774...
1787    C:\Users\markf\Desktop\PadChest\0\301634102198...
2264    C:\Users\markf\Desktop\PadChest\0\739243506699...
1334    C:\Users\markf\Desktop\PadChest\0\249891250835...
                              ...                        
1635    C:\Users\markf\Desktop\PadChest\0\263453479364...
497     C:\Users\markf\Desktop\PadChest\0\182707296104...
2720    C:\Users\markf\Desktop\PadChest\0\223994752115...
2379    C:\Users\markf\Desktop\PadChest\0\250538383336...
1267    C:\Users\markf\Desktop\PadChest\0\168222128175...
Name: Full_Image_Path, Length: 2162, dtype: object

In [29]:
# # Define the model
# model = tf.keras.Sequential([
#     tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(256, 256, 3)),
#     tf.keras.layers.MaxPooling2D((2, 2)),
#     tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
#     tf.keras.layers.MaxPooling2D((2, 2)),
#     tf.keras.layers.Flatten(),
#     tf.keras.layers.Dense(128, activation='relu'),
#     tf.keras.layers.Dense(len(class_to_index), activation='sigmoid')  # Sigmoid for multi-label classification
# ])

# # Compile the model
# model.compile(
#     optimizer='adam',
#     loss='binary_crossentropy',  # Multi-label classification loss
#     metrics=['accuracy']
# )


In [30]:
# # Train the model
# history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)

# # Evaluate on the test set
# test_loss, test_accuracy = model.evaluate(test_ds)
# print(f"Test Loss: {test_loss}")
# print(f"Test Accuracy: {test_accuracy}")

# # Save the model
# model.save('multi_label_xray_model.h5')

# # To visualize class indices
# print("Class-to-Index Mapping:", class_to_index)

In [31]:
# import matplotlib.pyplot as plt

# # Function to visualize predictions vs actual labels
# def visualize_predictions(model, dataset, class_names, num_images=3):
#     # Get a batch of data
#     for images, labels in dataset.take(1):
#         # Predict on the images
#         predictions = model.predict(images)
#         predicted_classes = tf.argmax(predictions, axis=1)
#         actual_classes = tf.argmax(labels, axis=1)

#         # Plot the images with predictions
#         plt.figure(figsize=(15, 15))
#         for i in range(num_images):
#             plt.subplot(1, num_images, i + 1)
#             plt.imshow(images[i].numpy())
#             plt.axis('off')
#             predicted_label = ", ".join([class_names[idx] for idx, pred in enumerate(predictions[i]) if pred > 0.5])
#             actual_label = ", ".join([class_names[idx] for idx in range(len(class_names)) if labels[i][idx]])
#             plt.title(f"P: {predicted_label}\nA: {actual_label}", fontsize=8)
#         plt.show()
#         break

# visualize_predictions(model, test_ds, list(class_to_index.keys()))


In [77]:
class_labels = list(class_to_index.keys())

# Build the Model
base_model = Xception(weights='imagenet', include_top=False, pooling='avg', input_shape=(256, 256, 3))
base_model.trainable = False

model = Sequential([
    base_model,
    BatchNormalization(),
    Dropout(0.45),
    Dense(220, activation='relu'),
    Dropout(0.25),
    Dense(60, activation='relu'),
    Dense(len(class_labels), activation='sigmoid')  # Output shape matches the number of classes
])

# Compile the model
model.compile(optimizer=Adamax(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])

# Model Summary
model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ xception (Functional)           │ (None, 2048)           │    20,861,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_29          │ (None, 2048)           │         8,192 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 220)            │       450,780 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 220)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 60)             │        13,260 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 162)            │         9,882 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,343,594 (81.42 MB)

 Trainable params: 478,018 (1.82 MB)

 Non-trainable params: 20,865,576 (79.60 MB)

In [78]:
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# training model
history = model.fit(
    train_ds,
    epochs=20,
    validation_data=val_ds,
    callbacks=[early_stopping]
)

Epoch 1/20
68/68 ━━━━━━━━━━━━━━━━━━━━ 144s 2s/step - accuracy: 0.0098 - loss: 0.5510 - val_accuracy: 0.3112 - val_loss: 0.0731
Epoch 2/20
68/68 ━━━━━━━━━━━━━━━━━━━━ 134s 2s/step - accuracy: 0.2725 - loss: 0.0636 - val_accuracy: 0.3112 - val_loss: 0.0607
Epoch 3/20
68/68 ━━━━━━━━━━━━━━━━━━━━ 146s 2s/step - accuracy: 0.2842 - loss: 0.0585 - val_accuracy: 0.3112 - val_loss: 0.0591
Epoch 4/20
68/68 ━━━━━━━━━━━━━━━━━━━━ 123s 2s/step - accuracy: 0.2929 - loss: 0.0576 - val_accuracy: 0.3112 - val_loss: 0.0585
Epoch 5/20
68/68 ━━━━━━━━━━━━━━━━━━━━ 123s 2s/step - accuracy: 0.2926 - loss: 0.0571 - val_accuracy: 0.3112 - val_loss: 0.0584
Epoch 6/20
68/68 ━━━━━━━━━━━━━━━━━━━━ 124s 2s/step - accuracy: 0.2957 - loss: 0.0571 - val_accuracy: 0.3112 - val_loss: 0.0583
Epoch 7/20
68/68 ━━━━━━━━━━━━━━━━━━━━ 123s 2s/step - accuracy: 0.2909 - loss: 0.0570 - val_accuracy: 0.3112 - val_loss: 0.0583
Epoch 8/20
68/68 ━━━━━━━━━━━━━━━━━━━━ 124s 2s/step - accuracy: 0.2899 - loss: 0.0569 - val_accuracy: 0.3112 - v

In [ ]:
# Evaluate the model on the validation dataset
validation_loss, validation_accuracy = model.evaluate(val_ds)

print("Validation Loss:", validation_loss)
print("Validation Accuracy:", validation_accuracy)

In [ ]:
# Find the epoch with the best validation accuracy
best_epoch = history.history['val_accuracy'].index(max(history.history['val_accuracy'])) + 1

# Plot Training and Validation Accuracy and Loss
plt.style.use('seaborn-darkgrid')
fig, axs = plt.subplots(1, 2, figsize=(16, 5))

# Training and Validation Accuracy
axs[0].plot(history.history['accuracy'], label='Training Accuracy', color='blue')
axs[0].plot(history.history['val_accuracy'], label='Validation Accuracy', color='red')
axs[0].scatter(best_epoch - 1, history.history['val_accuracy'][best_epoch - 1], color='green', label=f'Best Epoch: {best_epoch}')
axs[0].set_xlabel('Epoch')
axs[0].set_ylabel('Accuracy')
axs[0].set_title('Training and Validation Accuracy')
axs[0].legend()

# Training and Validation Loss
axs[1].plot(history.history['loss'], label='Training Loss', color='blue')
axs[1].plot(history.history['val_loss'], label='Validation Loss', color='red')
axs[1].scatter(best_epoch - 1, history.history['val_loss'][best_epoch - 1], color='green', label=f'Best Epoch: {best_epoch}')
axs[1].set_xlabel('Epoch')
axs[1].set_ylabel('Loss')
axs[1].set_title('Training and Validation Loss')
axs[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Visualize Model Predictions on Test Set
def plot_images_with_predictions(model, dataset, class_labels, num_images=20, num_images_per_row=5):
    # Get predictions
    dataset_batch = dataset.unbatch().batch(num_images)
    for images, labels in dataset_batch.take(1):
        predictions = model.predict(images)
        plt.figure(figsize=(15, 10))
        
        for i in range(num_images):
            predicted_labels = [class_labels[idx] for idx, pred in enumerate(predictions[i]) if pred > 0.5]
            actual_labels = [class_labels[idx] for idx, actual in enumerate(labels[i]) if actual == 1]
            
            plt.subplot(num_images // num_images_per_row + 1, num_images_per_row, i + 1)
            plt.imshow(images[i].numpy().astype("uint8"))
            plt.title(f"True: {', '.join(actual_labels)}\nPred: {', '.join(predicted_labels)}", fontsize=8)
            plt.axis('off')

        plt.tight_layout()
        plt.show()

# Visualize predictions on the test dataset
plot_images_with_predictions(model, test_ds, class_labels, num_images=20)